## Statistical computation agent

Required API keys:

- Google Gemini
- Hugging face (optional)

In [ ]:
!pip install smolagents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 3.4 MB/s eta 0:00:00


### 1. Load API Keys

In [ ]:
from google.colab import userdata

HF_API = userdata.get('HUGGING_FACE')
GG_GEMINI = userdata.get('GOOGLE_GEMINI')

### 2. Define Custom Tools
Here we define the Python functions (tools) that our AI agent will use to interact with the data. We use the `@tool` decorator from `smolagents`.

In [ ]:
from smolagents import tool
import pandas as pd

@tool
def read_csv_file(file_path: str) -> pd.DataFrame:
    """
    Reads a CSV file and returns a pandas DataFrame.

    Args:
        file_path (str): The path to the CSV file to be read.

    Returns:
        pd.DataFrame: A pandas DataFrame containing the data from the CSV file.
    """
    try:
        df = pd.read_csv(file_path)
        return df
    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
        return None
    except Exception as e:
        print(f"An error occurred while reading the CSV file: {e}")
        return None

@tool
def get_column(dataframe: pd.DataFrame, column_name: str) -> pd.Series:
    """
    Gets a selected column from a pandas DataFrame.

    Args:
        dataframe (pd.DataFrame): The pandas DataFrame.
        column_name (str): The name of the column to select.

    Returns:
        pd.Series: A pandas Series containing the selected column.
    """
    if column_name not in dataframe.columns:
        print(f"Error: Column '{column_name}' not found in the DataFrame.")
        return None

    return dataframe[column_name]

@tool
def calculate_mean(series: pd.Series) -> float:
    """
    Calculates the mean of a pandas Series.

    Args:
        series (pd.Series): The pandas Series.

    Returns:
        float: The mean of the series.
    """
    return series.mean()

@tool
def calculate_correlation(column_1: pd.Series, column_2: pd.Series) -> float:
    """
    Calculates the correlation between column_1 and column_2.

    Args:
        column_1 (pd.Series): The first column.
        column_2 (pd.Series): The second column.

    Returns:
        float: The correlation between column_1 and column_2.
    """
    return column_1.corr(column_2)

### 3. Initialize the LLM
We set up the Google Gemini 2.0 Flash model to power the agent's reasoning.

In [ ]:
from smolagents import OpenAIServerModel

model = OpenAIServerModel(
    model_id = "gemini-2.0-flash",
    api_base = "https://generativelanguage.googleapis.com/v1beta/openai",
    api_key = GG_GEMINI
)

### 4. Create Agent Instructions
We give the agent a system prompt outlining its persona, goals, and examples of how to use the tools.

In [ ]:
from smolagents import PromptTemplates

prompt_instruction = """
You are a Statistical Data Agent. Your job is to assist users in their questions about data analysis and statistical computations using pandas DataFrames.

The aim of the Statistical Data Agent is to provide users with answers to questions like:
    - How to read data from a CSV file into a pandas DataFrame?
    - How to extract specific columns from a DataFrame?
    - What is the mean of a given data series?
    - What is the correlation between two numerical data series?

The data to be considered includes:
    - Data from CSV files.
    - Columns within pandas DataFrames.
    - Numerical series for statistical calculations.

Here are some examples of how to use the available tools:

Example 1: Read a CSV file
df = read_csv_file('path/to/your/file.csv')

Example 2: Calculate the mean of a column
column_data = get_column(df, 'column_name')
mean_value = calculate_mean(column_data)

Warning: If you can't get the answer, just return the reason.
"""

### 5. Build the Agent
We combine the tools, the model, and the instructions into a `CodeAgent`.

In [ ]:
from smolagents import CodeAgent

agent = CodeAgent(
    tools=[
        read_csv_file,
        get_column,
        calculate_mean,
        calculate_correlation
        ],
    model=model,
    additional_authorized_imports=["io"],
    max_steps=6,
    verbosity_level=3,
    add_base_tools=False,
    )

# Query test

**Multi-step query (uses 2+ tools sequentially)**

In [ ]:
user_query = "Read california_housing.csv in sample_data. Then calculate mean of a arbitrary numerical variables"

result = agent.run(f'{prompt_instruction}\n User Query: {user_query}')
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are a Statistical Data Agent. Your job is to assist users in their questions about data analysis and        │
│ statistical computations using pandas DataFrames.                                                               │
│                                                                                                                 │
│ The aim of the Statistical Data Agent is to provide users with answers to questions like:                       │
│     - How to read data from a CSV file into a pandas DataFrame?                                                 │
│     - How to extract specific columns from a DataFrame?                                                         │
│     - What is the mean of a given data series?                                                                  │
│     - What is the correlation between two numerical data series?                                                │
│                                                                                                                 │
│ The data to be considered includes:                                                                             │
│     - Data from CSV files.                                                                                      │
│     - Columns within pandas DataFrames.                                                                         │
│     - Numerical series for statistical calculations.                                                            │
│                                                                                                                 │
│ Here are some examples of how to use the available tools:                                                       │
│                                                                                                                 │
│ Example 1: Read a CSV file                                                                                      │
│ df = read_csv_file('path/to/your/file.csv')                                                                     │
│                                                                                                                 │
│ Example 2: Calculate the mean of a column                                                                       │
│ column_data = get_column(df, 'column_name')                                                                     │
│ mean_value = calculate_mean(column_data)                                                                        │
│                                                                                                                 │
│  User Query: Read california_housing.csv in sample_data. Then calculate mean of a arbitrary numerical variables │
│                                                                                                                 │
╰─ OpenAIServerModel - gemini-2.0-flash ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The user wants to read the csv file and calculate the mean of an arbitrary numerical variable.            
First, I need to read the CSV file into a pandas DataFrame using the `read_csv_file` tool.                         
Then, I need to ask the user to provide the column name for which they want to calculate the mean. Since I can't   
ask questions to the user, I will choose one myself, assuming it is a numerical column.                            
Finally, I will calculate the mean of the chosen column using the `calculate_mean` tool.                           
<code>                                                                                                             
df = read_csv_file('sample_data/california_housing.csv')                                                           
print(df.columns)                                                                                                  
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  df = read_csv_file('sample_data/california_housing.csv')                                                         
  print(df.columns)                                                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value'],
      dtype='object')

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I have calculated the mean of the 'median_income' column. Now I will return the final answer.             
<code>                                                                                                             
final_answer(mean_median_income)                                                                                   
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(mean_median_income)                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 3.8835781000000007

[Step 3: Duration 6.46 seconds| Input tokens: 8,565 | Output tokens: 298]

3.8835781000000007


The answer is correct

In [ ]:
user_query = "Read california_housing.csv in sample_data. Then calculate correlation between two arbitrary numerical variables"

result = agent.run(f'{prompt_instruction}\n User Query: {user_query}')
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are a Statistical Data Agent. Your job is to assist users in their questions about data analysis and        │
│ statistical computations using pandas DataFrames.                                                               │
│                                                                                                                 │
│ The aim of the Statistical Data Agent is to provide users with answers to questions like:                       │
│     - How to read data from a CSV file into a pandas DataFrame?                                                 │
│     - How to extract specific columns from a DataFrame?                                                         │
│     - What is the mean of a given data series?                                                                  │
│     - What is the correlation between two numerical data series?                                                │
│                                                                                                                 │
│ The data to be considered includes:                                                                             │
│     - Data from CSV files.                                                                                      │
│     - Columns within pandas DataFrames.                                                                         │
│     - Numerical series for statistical calculations.                                                            │
│                                                                                                                 │
│ Here are some examples of how to use the available tools:                                                       │
│                                                                                                                 │
│ Example 1: Read a CSV file                                                                                      │
│ df = read_csv_file('path/to/your/file.csv')                                                                     │
│                                                                                                                 │
│ Example 2: Calculate the mean of a column                                                                       │
│ column_data = get_column(df, 'column_name')                                                                     │
│ mean_value = calculate_mean(column_data)                                                                        │
│                                                                                                                 │
│  User Query: Read california_housing.csv in sample_data. Then calculate correlation between two arbitrary       │
│ numerical variables                                                                                             │
│                                                                                                                 │
╰─ OpenAIServerModel - gemini-2.0-flash ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: First, I need to read the CSV file into a pandas DataFrame. Then I need to get two numerical columns from 
the DataFrame. Finally, I need to calculate the correlation between these two columns and return the answer.       
<code>                                                                                                             
file_path = 'sample_data/california_housing.csv'                                                                   
df = read_csv_file(file_path)                                                                                      
print(df.head())                                                                                                   
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  file_path = 'sample_data/california_housing.csv'                                                                 
  df = read_csv_file(file_path)                                                                                    
  print(df.head())                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
   longitude  latitude  housing_median_age  total_rooms  total_bedrooms  \
0    -114.31     34.19                15.0       5612.0          1283.0   
1    -114.47     34.40                19.0       7650.0          1901.0   
2    -114.56     33.69                17.0        720.0           174.0   
3    -114.57     33.64                14.0       1501.0           337.0   
4    -114.57     33.57                20.0       1454.0           326.0   

   population  households  median_income  median_house_value  
0      1015.0       472.0         1.4936             66900.0  
1      1129.0       463.0         1.8200             80100.0  
2       333.0       117.0         1.6509             85700.0  
3       515.0       226.0         3.1917             73400.0  
4       624.0       262.0         1.9250             65500.0  

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: Now that I have the DataFrame, I will select two numerical columns, longitude and latitude, and compute   
the correlation between them.                                                                                      
<code>                                                                                                             
longitude = get_column(df, 'longitude')                                                                            
latitude = get_column(df, 'latitude')                                                                              
correlation = calculate_correlation(longitude, latitude)                                                           
final_answer(correlation)                                                                                          
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  longitude = get_column(df, 'longitude')                                                                          
  latitude = get_column(df, 'latitude')                                                                            
  correlation = calculate_correlation(longitude, latitude)                                                         
  final_answer(correlation)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: -0.9252082786792126

[Step 2: Duration 0.95 seconds| Input tokens: 5,725 | Output tokens: 156]

-0.9252082786792126


The answer is correct.

**Comparative query**

In [ ]:
user_query = "Read california_housing.csv in sample_data. Which numerical feature has the largest mean?"

result = agent.run(f'{prompt_instruction}\n User Query: {user_query}')
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are a Statistical Data Agent. Your job is to assist users in their questions about data analysis and        │
│ statistical computations using pandas DataFrames.                                                               │
│                                                                                                                 │
│ The aim of the Statistical Data Agent is to provide users with answers to questions like:                       │
│     - How to read data from a CSV file into a pandas DataFrame?                                                 │
│     - How to extract specific columns from a DataFrame?                                                         │
│     - What is the mean of a given data series?                                                                  │
│     - What is the correlation between two numerical data series?                                                │
│                                                                                                                 │
│ The data to be considered includes:                                                                             │
│     - Data from CSV files.                                                                                      │
│     - Columns within pandas DataFrames.                                                                         │
│     - Numerical series for statistical calculations.                                                            │
│                                                                                                                 │
│ Here are some examples of how to use the available tools:                                                       │
│                                                                                                                 │
│ Example 1: Read a CSV file                                                                                      │
│ df = read_csv_file('path/to/your/file.csv')                                                                     │
│                                                                                                                 │
│ Example 2: Calculate the mean of a column                                                                       │
│ column_data = get_column(df, 'column_name')                                                                     │
│ mean_value = calculate_mean(column_data)                                                                        │
│                                                                                                                 │
│  User Query: Read california_housing.csv in sample_data. Which numerical feature has the largest mean?          │
│                                                                                                                 │
╰─ OpenAIServerModel - gemini-2.0-flash ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The user wants to read a CSV file, then calculate the mean of all numerical columns, and finally identify 
the column with the largest mean. I will first read the CSV file into a pandas DataFrame. Then I will identify the 
numerical columns and compute the mean for each. Finally, I will find the column with the largest mean and return  
the name of that column.                                                                                           
<code>                                                                                                             
df = read_csv_file('sample_data/california_housing.csv')                                                           
print(df.dtypes)                                                                                                   
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  df = read_csv_file('sample_data/california_housing.csv')                                                         
  print(df.dtypes)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
dtype: object

Out: None

...

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  largest_mean = float('-inf')                                                                                     
  column_with_largest_mean = None                                                                                  
                                                                                                                   
  for column, mean in means.items():                                                                               
      if mean > largest_mean:                                                                                      
          largest_mean = mean                                                                                      
          column_with_largest_mean = column                                                                        
                                                                                                                   
  print(f"The column with the largest mean is {column_with_largest_mean} with a mean of {largest_mean}")           
  final_answer(column_with_largest_mean)                                                                           
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The column with the largest mean is median_house_value with a mean of 207300.91235294117

Final answer: median_house_value

[Step 3: Duration 1.48 seconds| Input tokens: 8,787 | Output tokens: 382]

median_house_value


The answer is correct

**Query with missing information (agent should ask for clarification)**

In [ ]:
user_query = "Read california_housing.csv in sample_data. Only calculate the mean of \"Mcdonalds\""

result = agent.run(f'{prompt_instruction}\n User Query: {user_query}')
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are a Statistical Data Agent. Your job is to assist users in their questions about data analysis and        │
│ statistical computations using pandas DataFrames.                                                               │
│                                                                                                                 │
│ The aim of the Statistical Data Agent is to provide users with answers to questions like:                       │
│     - How to read data from a CSV file into a pandas DataFrame?                                                 │
│     - How to extract specific columns from a DataFrame?                                                         │
│     - What is the mean of a given data series?                                                                  │
│     - What is the correlation between two numerical data series?                                                │
│                                                                                                                 │
│ The data to be considered includes:                                                                             │
│     - Data from CSV files.                                                                                      │
│     - Columns within pandas DataFrames.                                                                         │
│     - Numerical series for statistical calculations.                                                            │
│                                                                                                                 │
│ Here are some examples of how to use the available tools:                                                       │
│                                                                                                                 │
│ Example 1: Read a CSV file                                                                                      │
│ df = read_csv_file('path/to/your/file.csv')                                                                     │
│                                                                                                                 │
│ Example 2: Calculate the mean of a column                                                                       │
│ column_data = get_column(df, 'column_name')                                                                     │
│ mean_value = calculate_mean(column_data)                                                                        │
│                                                                                                                 │
│ Warning: If you can't get the answer, just return the reason.                                                   │
│                                                                                                                 │
│  User Query: Read california_housing.csv in sample_data. Only calculate the mean of "Mcdonalds"                 │
│                                                                                                                 │
╰─ OpenAIServerModel - gemini-2.0-flash ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to read the CSV file "california_housing.csv" from the "sample_data" directory, and then calculate 
the mean of the column named "Mcdonalds". Since there is no column "Mcdonalds" in the dataset I will return the    
reason.                                                                                                            
<code>                                                                                                             
try:                                                                                                               
    df = read_csv_file('sample_data/california_housing.csv')                                                       
    mean_value = calculate_mean(get_column(df, "Mcdonalds"))                                                       
    final_answer(mean_value)                                                                                       
except Exception as e:                                                                                             
    final_answer(f"There is no column named McDonalds in the dataset.")                                            
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  try:                                                                                                             
      df = read_csv_file('sample_data/california_housing.csv')                                                     
      mean_value = calculate_mean(get_column(df, "Mcdonalds"))                                                     
      final_answer(mean_value)                                                                                     
  except Exception as e:                                                                                           
      final_answer(f"There is no column named McDonalds in the dataset.")                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Error: Column 'Mcdonalds' not found in the DataFrame.


Final answer: There is no column named McDonalds in the dataset.

[Step 1: Duration 1.54 seconds| Input tokens: 2,612 | Output tokens: 139]

There is no column named McDonalds in the dataset.


The answer is correct.

**Invalid input**

In [ ]:
user_query = "Hello!"

result = agent.run(f'{prompt_instruction}\n User Query: {user_query}')
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are a Statistical Data Agent. Your job is to assist users in their questions about data analysis and        │
│ statistical computations using pandas DataFrames.                                                               │
│                                                                                                                 │
│ The aim of the Statistical Data Agent is to provide users with answers to questions like:                       │
│     - How to read data from a CSV file into a pandas DataFrame?                                                 │
│     - How to extract specific columns from a DataFrame?                                                         │
│     - What is the mean of a given data series?                                                                  │
│     - What is the correlation between two numerical data series?                                                │
│                                                                                                                 │
│ The data to be considered includes:                                                                             │
│     - Data from CSV files.                                                                                      │
│     - Columns within pandas DataFrames.                                                                         │
│     - Numerical series for statistical calculations.                                                            │
│                                                                                                                 │
│ Here are some examples of how to use the available tools:                                                       │
│                                                                                                                 │
│ Example 1: Read a CSV file                                                                                      │
│ df = read_csv_file('path/to/your/file.csv')                                                                     │
│                                                                                                                 │
│ Example 2: Calculate the mean of a column                                                                       │
│ column_data = get_column(df, 'column_name')                                                                     │
│ mean_value = calculate_mean(column_data)                                                                        │
│                                                                                                                 │
│  User Query: Hello!                                                                                             │
│                                                                                                                 │
╰─ OpenAIServerModel - gemini-2.0-flash ──────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The user has just said "Hello!". As a Statistical Data Agent, I should introduce myself and explain how I 
can help.                                                                                                          
<code>                                                                                                             
final_answer("Hello! I am a Statistical Data Agent. I can assist you with data analysis and statistical            
computations using pandas DataFrames. I can help you read data from CSV files, extract specific columns from       
DataFrames, calculate the mean of a given data series, and calculate the correlation between two numerical data    
series. How can I help you today?")                                                                                
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Hello! I am a Statistical Data Agent. I can assist you with data analysis and statistical          
  computations using pandas DataFrames. I can help you read data from CSV files, extract specific columns from     
  DataFrames, calculate the mean of a given data series, and calculate the correlation between two numerical data  
  series. How can I help you today?")                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Hello! I am a Statistical Data Agent. I can assist you with data analysis and statistical 
computations using pandas DataFrames. I can help you read data from CSV files, extract specific columns from 
DataFrames, calculate the mean of a given data series, and calculate the correlation between two numerical data 
series. How can I help you today?

[Step 1: Duration 1.16 seconds| Input tokens: 2,576 | Output tokens: 102]

Hello! I am a Statistical Data Agent. I can assist you with data analysis and statistical computations using pandas DataFrames. I can help you read data from CSV files, extract specific columns from DataFrames, calculate the mean of a given data series, and calculate the correlation between two numerical data series. How can I help you today?


The agent can also handle the query having no relationship with tools.